# Step 9 - Feature Batches with InceptionV3 (extension)

This step builds `.npz` batches using **InceptionV3** as the feature extractor. InceptionV3
produces **2048-dimensional** features and requires a **299x299** input (versus 224x224 for
VGG-16 and MobileNetV2).

Only the feature extractor and input size change; object detection (Faster R-CNN), the random 3-5 accident composition, and the `.npz` format are the same as Step 6.

**Compatibility:** feature length 2048 is incompatible with the author's pretrained model
(which expects 4096). The model's input size must be set to 2048 and the model retrained.

- `data`   : `(10, 100, 20, 2048)` float16
- `labels` : `(10, 2)` float16
- output   : `Dataset/features_inception/<split>/`

## 1. Models, helpers, and per-video encoder
InceptionV3's final classifier is replaced with identity so the network outputs the 2048-d
pooled features.

In [1]:
import cv2, time, glob, os
import numpy as np
import torch
import torch.nn as nn

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

N_FRAMES=100; N_DETECTION=20; N_OBJECTS=19
FEAT_DIM=2048          # <-- InceptionV3 feature length
IMG_SIZE=299           # <-- InceptionV3 input size (not 224)
KEEP={1,2,3,4,6,7,8}; SCORE_THRESH=0.5
MEAN=torch.tensor([0.485,0.456,0.406]).view(1,3,1,1).to(device)
STD =torch.tensor([0.229,0.224,0.225]).view(1,3,1,1).to(device)

# --- detector ---
from torchvision.models.detection import fasterrcnn_resnet50_fpn
try:
    from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights
    detector = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)
except Exception:
    detector = fasterrcnn_resnet50_fpn(pretrained=True)
detector.eval().to(device)

# --- InceptionV3 feature extractor (fc replaced by identity -> 2048 pooled features) ---
from torchvision.models import inception_v3
try:
    from torchvision.models import Inception_V3_Weights
    incep = inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1)
except Exception:
    incep = inception_v3(pretrained=True)
incep.fc = nn.Identity()       # output the 2048-d pooled features
incep.aux_logits = False        # no auxiliary head needed for inference
incep.eval().to(device)

def detect_boxes(frame):
    t=torch.from_numpy(frame).permute(2,0,1).float().to(device)/255.0
    with torch.no_grad():
        out=detector([t])[0]
    b=out['boxes'].cpu().numpy(); l=out['labels'].cpu().numpy(); s=out['scores'].cpu().numpy()
    kept=[(float(sc),bx) for bx,lb,sc in zip(b,l,s) if sc>=SCORE_THRESH and int(lb) in KEEP]
    kept.sort(key=lambda z:z[0], reverse=True)
    return [bx for _,bx in kept[:N_OBJECTS]]

def crop_box(frame, box):
    h,w,_=frame.shape; x1,y1,x2,y2=box
    x1=max(0,int(round(x1))); y1=max(0,int(round(y1)))
    x2=min(w,int(round(x2))); y2=min(h,int(round(y2)))
    return None if (x2-x1<2 or y2-y1<2) else frame[y1:y2,x1:x2]

def to_features_batch(crops):
    if not crops: return np.zeros((0,FEAT_DIM),np.float32)
    batch=np.stack([cv2.resize(c,(IMG_SIZE,IMG_SIZE)) for c in crops]).astype(np.float32)/255.0
    x=torch.from_numpy(batch).permute(0,3,1,2).to(device)
    x=(x-MEAN)/STD
    with torch.no_grad():
        x=incep(x)                          # (N, 2048) because fc is Identity
    return x.cpu().numpy().astype(np.float32)

def encode_video(path, max_frames=100):
    cap=cv2.VideoCapture(path); frames=[]
    while True:
        ok,f=cap.read()
        if not ok: break
        frames.append(cv2.cvtColor(f,cv2.COLOR_BGR2RGB))
    cap.release()
    out=np.zeros((N_FRAMES,N_DETECTION,FEAT_DIM),np.float32)
    for t in range(min(max_frames,len(frames))):
        frame=frames[t]; crops=[frame]
        for box in detect_boxes(frame):
            c=crop_box(frame,box)
            if c is not None: crops.append(c)
            if len(crops)==N_DETECTION: break
        feats=to_features_batch(crops)
        out[t,:feats.shape[0],:]=feats
    return out
print('InceptionV3 pipeline ready. FEAT_DIM =', FEAT_DIM, ' input =', IMG_SIZE)

device: cpu


Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to C:\Users\niles/.cache\torch\hub\checkpoints\inception_v3_google-0cc3c7bd.pth


A Jupyter Widget

InceptionV3 pipeline ready. FEAT_DIM = 2048  input = 299


## 2. Batch composition (random 3-5 accidents per batch)

In [3]:
# ----------------- CONFIG -----------------
VIDEOS_ROOT = '../Dataset/videos'   # local; on Kaggle: /kaggle/input/<dataset>/videos
SPLIT       = 'test'
N_BATCHES   = 1
MAX_FRAMES  = 20        # 20 = quick test, 100 = full
OUT_ROOT    = '../Dataset/features_inception'
ACC_MIN, ACC_MAX = 3, 5   # accidents per batch (the rest are non-accident)
SEED = 1234               # fixed seed -> reproducible AND resumable
# ------------------------------------------

import random

acc_pool = sorted(glob.glob(f'{VIDEOS_ROOT}/{SPLIT}/Accident/*.mp4'))
non_pool = sorted(glob.glob(f'{VIDEOS_ROOT}/{SPLIT}/NonAccident/*.mp4'))
print(f'{SPLIT}: {len(acc_pool)} accident, {len(non_pool)} non-accident')
assert acc_pool and non_pool, 'no videos - fix VIDEOS_ROOT'

def _n_acc(b):
    return random.Random(SEED + b).randint(ACC_MIN, ACC_MAX)

def plan_batch(b):                    # 0-based; deterministic so reruns/resume reproduce it
    rng   = random.Random(SEED + b)
    n_acc = rng.randint(ACC_MIN, ACC_MAX)
    n_non = 10 - n_acc
    acc_off = sum(_n_acc(j) for j in range(b))
    non_off = sum(10 - _n_acc(j) for j in range(b))
    acc = [acc_pool[(acc_off + k) % len(acc_pool)] for k in range(n_acc)]
    non = [non_pool[(non_off + k) % len(non_pool)] for k in range(n_non)]
    items = [(p, (0.0, 1.0)) for p in acc] + [(p, (1.0, 0.0)) for p in non]
    rng.shuffle(items)
    return items

test: 165 accident, 301 non-accident


## 3. Build + save (resumable: existing batches are skipped)

In [4]:
out_dir=f'{OUT_ROOT}/{SPLIT}'; os.makedirs(out_dir, exist_ok=True)
for b in range(1, N_BATCHES+1):
    out_path=f'{out_dir}/batch_{b:03d}.npz'
    if os.path.exists(out_path):
        print(f'skip batch_{b:03d} (exists)'); continue
    items=plan_batch(b-1)
    data=np.zeros((10,N_FRAMES,N_DETECTION,FEAT_DIM),np.float32)
    labels=np.zeros((10,2),np.float32)
    t0=time.time(); print(f'=== batch_{b:03d} ===')
    for i,(path,lab) in enumerate(items):
        ti=time.time(); data[i]=encode_video(path,MAX_FRAMES); labels[i]=lab
        print(f'  [{i+1:2d}/10] {"A" if lab[1]==1 else "n"} {os.path.basename(path):12s} {time.time()-ti:5.1f}s')
    np.savez(out_path, data=data.astype(np.float16), labels=labels.astype(np.float16))
    print(f'  saved {out_path}  ({os.path.getsize(out_path)/1e6:.1f} MB)  total {time.time()-t0:.1f}s\n')
print('DONE.')

=== batch_001 ===
  [ 1/10] n 000830.mp4    11.7s
  [ 2/10] n 000831.mp4    14.3s
  [ 3/10] n 000832.mp4    14.0s
  [ 4/10] A 000456.mp4    10.7s
  [ 5/10] A 000457.mp4    10.8s
  [ 6/10] A 000458.mp4    13.9s
  [ 7/10] n 000833.mp4    12.2s
  [ 8/10] n 000834.mp4    13.2s
  [ 9/10] A 000459.mp4    13.7s
  [10/10] A 000460.mp4    15.7s
  saved ../Dataset/features_inception/test/batch_001.npz  (81.9 MB)  total 130.4s

DONE.


## 4. Verify

In [5]:
f=np.load(f'{out_dir}/batch_001.npz')
print('data  :', f['data'].shape, f['data'].dtype, '  (feature length should be 2048)')
print('labels:', f['labels'].shape)
print('order :', ['A' if r[1]==1 else 'n' for r in f['labels']])
print('data >= 0 ?', bool((f['data']>=0).all()), ' max:', round(float(f['data'].max()),2))

data  : (10, 100, 20, 2048) float16   (feature length should be 2048)
labels: (10, 2)
order : ['n', 'n', 'n', 'A', 'A', 'A', 'n', 'n', 'A', 'A']
data >= 0 ? True  max: 8.77


## Summary
- Identical pipeline to Step 6, with InceptionV3 replacing VGG-16 and a 299x299 input.
- Feature length is **2048**; batch format and composition are otherwise unchanged.
- Requires retraining the DSA-RNN with `n_input = 2048`; not compatible with the author's
  pretrained weights.
- Together with Steps 6 (VGG-16) and 8 (MobileNetV2), this supports a backbone comparison:
  accuracy (AP), earliness (Time-to-Accident), and efficiency (speed, model size).